# BiHPR Usage Demo

This notebook demonstrates the standard workflow for the BiHPR package:

1. import the public API;
2. generate a small paper-style simulation dataset;
3. fit one fixed penalty setting;
4. run a warm-started BIC tuning path;
5. compute recovery metrics and inspect the tuning table.

The example uses a small dataset and short iteration limits so it can be used as a quick package check. Increase the grid size and iteration limits for serious experiments.

## 1. Imports

Install the package from the repository root before running this notebook:

```bash
pip install -e ".[examples]"
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from BiHPR import (
    __version__,
    fit_bihpr_mcp_large,
    fit_bihpr_path,
    generate_paper_simulation,
)

print("BiHPR version:", __version__)

## 2. Generate a Simulation Dataset

`generate_paper_simulation` creates a balanced design with three sample subgroups, three active feature clusters, and one inactive zero-feature cluster. Here we use a small problem for speed.

In [ ]:
truth = generate_paper_simulation(
    n=30,
    p=60,
    p0=15,
    sigma=0.5,
    seed=2026,
)

Y = truth["Y"]
X = truth["X"]

print("Y shape:", Y.shape)
print("X shape:", X.shape)
print("True beta shape:", truth["beta_true"].shape)
print("Number of active features:", int(truth["active_features"].sum()))

## 3. Fit One Penalty Setting

Use `fit_bihpr_mcp_large` when you already know the structural penalty (`lambda_col`) and sparsity penalty (`lambda3`). The solver returns the estimated coefficient matrix, cluster labels, active-feature labels, convergence diagnostics, and timing information.

In [ ]:
single = fit_bihpr_mcp_large(
    Y,
    X,
    lambda_col=2000.0,
    lambda3=500.0,
    niter=80,
    tol=1e-3,
    cluster_tol=1e-3,
)

summary = {
    "iterations": single["iterations"],
    "converged": single["converged"],
    "rss": single["rss"],
    "n_active": int(single["active_features"].sum()),
    "n_row_clusters": len(np.unique(single["row_labels"])),
    "n_col_clusters": len(np.unique(single["col_labels"])),
}
summary

## 4. Run a BIC Tuning Path

`fit_bihpr_path` first runs pilot models with `lambda3=0`, computes adaptive feature weights, and then searches the two-dimensional tuning grid with warm starts.

In [ ]:
path = fit_bihpr_path(
    Y,
    X,
    lambda_grid=(4000.0, 2000.0, 1000.0),
    lambda3_grid=(1000.0, 500.0, 100.0),
    niter=80,
    pilot_niter=80,
    tol=1e-3,
    cluster_tol=1e-3,
    feature_threshold=0.9,
    label_cluster_threshold=5.0,
)

path["best_record"]

## 5. Inspect the Tuning Table

In [ ]:
records = pd.DataFrame(path["records"])
records.sort_values("bic").head()

## 6. Compute Recovery Metrics

The exact numbers will vary with the tuning grid and iteration limits. For publication-quality experiments, use larger grids and stricter tolerances.

In [ ]:
best = path["best"]
true_active = truth["active_features"]
estimated_active = best["active_features"]

metrics = {
    "row_ari": adjusted_rand_score(truth["row_labels"], best["row_labels"]),
    "active_col_ari": adjusted_rand_score(
        truth["col_labels"][true_active],
        best["col_labels"][true_active],
    ),
    "fpr": float(np.mean(estimated_active[~true_active])),
    "fnr": float(np.mean(~estimated_active[true_active])),
    "n_active": int(estimated_active.sum()),
    "rmse": float(
        np.sqrt(np.mean((truth["Y"] - np.einsum("ij,ij->i", truth["X"], best["beta"])) ** 2))
    ),
}

metrics

## 7. Next Steps

- Increase `niter` and `pilot_niter` for stable convergence.
- Expand `lambda_grid` and `lambda3_grid` for robust tuning.
- Use the command-line script for multi-dimensional Gaussian experiments:

```bash
bihpr-gaussian-grid --grid pilot --p-values 100 300 --repeats 5 --n-jobs 2
```